# 0.4 · Polars Intro 入门

> **课程定位 / Where this fits**
> 第 4 课，**Part 0 · 基础准备**。
> Lesson 4, **Part 0 · Foundations**.
>
> Pandas 已经统治 DS 工具圈十多年了。**Polars** 是 2020 年后用 Rust 重写的现代 DataFrame，**比 pandas 快 5–30 倍、内存少一半、API 更一致**。大厂招聘也越来越多看到它。
> Pandas has ruled DS tooling for over a decade. **Polars** (2020+) is a Rust-built modern DataFrame — **5–30× faster than pandas, half the memory, more consistent API**. Increasingly common in job postings.

> 📐 **符号约定 / Notation**（见 [`NOTATION.md`](../NOTATION.md)）
> 本节几乎都是数据操作，数学符号少。$n$ = 行数，$d$ = 列数。
> Mostly data manipulation. $n$ = rows, $d$ = columns.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 解释 Polars 比 pandas 快的**三个根本原因**：列存（Arrow）+ 多线程（Rust）+ 惰性执行（query optimizer）。
   Explain the three reasons Polars is faster: columnar (Arrow) + multi-threaded (Rust) + lazy with optimizer.
2. 掌握 **Expression API**（`pl.col(...)`）—— Polars 区别于 pandas 的核心设计。
   Master the **Expression API** — the key design difference vs pandas.
3. 区分 **eager** 与 **lazy** 两种模式，知道何时用哪种。
   Tell apart eager and lazy modes; know when to use each.
4. 用 `select` / `filter` / `with_columns` / `group_by` / `join` 完成绝大多数数据操作。
   Do most data ops with `select`, `filter`, `with_columns`, `group_by`, `join`.
5. 用**窗口函数 `over`** 表达"按组排名/累计"等 SQL 风格分析。
   Express "rank/cumsum per group" with **window functions (`over`)**.
6. 在 **1,000,000 行**合成数据上实测 Polars vs pandas 性能差距。
   Benchmark Polars vs pandas on **1M rows**.

---

## 目录 / Table of Contents

1. [为什么是 Polars / Why Polars](#1)
2. [🛒 数据集介绍：合成 E-commerce 订单 / Synthetic E-commerce Orders](#2)
3. [Eager DataFrame 基础 / Eager DataFrame Basics](#3)
4. [Expression API —— 核心 / Expression API — the Core](#4)
5. [`select` / `with_columns` / `filter`](#5)
6. [聚合与分组 / Aggregations & `group_by`](#6)
7. [窗口函数 / Window functions with `over`](#7)
8. [连接 / Joins](#8)
9. [长宽转换 / `pivot` and `unpivot`](#9)
10. [字符串与日期 / Strings and Datetimes](#10)
11. [Lazy API + Query Plan 优化](#11)
12. [SQL on Polars](#12)
13. [性能对比 pandas / Benchmark vs Pandas](#13)
14. [实战：订单数据 EDA / Hands-on EDA](#14)
15. [小结 / Summary](#15)


<a id="1"></a>
## 1. 为什么是 Polars / Why Polars

| 维度 / Aspect | pandas | Polars |
|---|---|---|
| 后端 / Backend | Python + NumPy (C) | **Rust + Apache Arrow** |
| 内存布局 / Layout | 行 + 列混合 / row-ish | **列存 / columnar** |
| 并行 / Parallel | 单线程默认 / single-thread | **多线程默认 / multi-threaded** |
| 执行模式 / Mode | eager only | **eager + lazy** |
| 查询优化 / Query optimizer | ❌ | ✅ 谓词下推、投影下推、公共子表达式消除 |
| 缺失值 / Missing data | `NaN`（破坏整数类型） | `null`（保留 dtype） |
| API 一致性 / API consistency | 历史包袱重 / legacy quirks | **表达式 API 统一** |

**核心思想 / Core idea**：
- **Expression API** —— `pl.col("price") * pl.col("qty")` 是一个**计算图节点**，不是立即求值
- **Lazy** —— 整个查询攒到 `.collect()` 时一次性优化执行
- **Columnar + Rust** —— 多线程 + SIMD 暴打 Python

---

**何时用 / When to use**：
- 数据 100K – 100M 行 → Polars 几乎总是更快
- 想从 pandas 迁移 → API 学习曲线 1–2 天
- 上 PB 级 → Spark / Dask（后面 Part 21）


In [ ]:
import polars as pl
import pandas as pd
import numpy as np

print(f"polars : {pl.__version__}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")

# 给 polars 配置漂亮输出 / Pretty output config
pl.Config.set_tbl_rows(8)
pl.Config.set_tbl_cols(10)


<a id="2"></a>
## 2. 🛒 数据集介绍：合成 E-commerce 订单 / Synthetic E-commerce Orders

> 真实大数据集 GB 起步、下载慢、Kaggle 要登录。**这里我们用 numpy 现造 1,000,000 行假数据**，方便:
> - 演示 Polars 在大数据上的速度
> - 你可以随时调 `N` 来感受不同规模下的差异
>
> Synthetic 1,000,000-row dataset generated with NumPy — fast to demo, easy to scale.
>
> **Schema / 字段**:
>
> | 列 / Column | 类型 / Type | 含义 / Meaning |
> |---|---|---|
> | `order_id`        | int64 | 订单号（主键）/ unique order id |
> | `user_id`         | int32 | 用户 id |
> | `order_ts`        | datetime | 下单时间 / order timestamp |
> | `category`        | str / category | 商品类目 |
> | `unit_price`      | float64 | 单价 / unit price ($) |
> | `quantity`        | int32 | 数量 |
> | `payment_method`  | str / category | `credit` / `debit` / `paypal` / `applepay` |
> | `country`         | str / category | 用户国家（5 个值）/ user country |
>
> **任务 / Task**：本节用它演示数据操作，**不建模**。
> Used here for demo only — no modeling.


In [ ]:
# 生成 1,000,000 行假数据 / Generate 1M rows
rng = np.random.default_rng(seed=42)
N = 1_000_000

# 时间戳：过去 2 年内随机 / Random ts over the past 2 years
t0 = pd.Timestamp("2024-01-01")
seconds_offset = rng.integers(0, 2 * 365 * 24 * 3600, size=N)
timestamps = t0 + pd.to_timedelta(seconds_offset, unit="s")

categories = ["Electronics", "Clothing", "Books", "Home", "Beauty", "Toys"]
payments = ["credit", "debit", "paypal", "applepay"]
countries = ["US", "UK", "DE", "JP", "CA"]

df_pl = pl.DataFrame({
    "order_id":       np.arange(N, dtype=np.int64),
    "user_id":        rng.integers(1, 50_000, size=N, dtype=np.int32),
    "order_ts":       timestamps,
    "category":       rng.choice(categories, size=N),
    "unit_price":     np.round(rng.gamma(shape=2, scale=20, size=N), 2),  # 偏分布 / skewed
    "quantity":       rng.integers(1, 6, size=N, dtype=np.int32),
    "payment_method": rng.choice(payments, size=N, p=[0.5, 0.3, 0.15, 0.05]),
    "country":        rng.choice(countries, size=N, p=[0.4, 0.2, 0.15, 0.15, 0.10]),
})

print(f"shape : {df_pl.shape}")
print(f"size  : {df_pl.estimated_size('mb'):.1f} MB")
df_pl.head(5)


<a id="3"></a>
## 3. Eager DataFrame 基础 / Eager DataFrame Basics

"Eager" = 立即执行（和 pandas 一样）。最入门最舒服的模式。
"Eager" = execute immediately (like pandas). The friendliest entry point.


In [ ]:
# 类似 pandas 的"看一眼" / pandas-like inspection
print("shape  :", df_pl.shape)
print("columns:", df_pl.columns)
print("dtypes :", df_pl.dtypes)
print("schema :", df_pl.schema)


In [ ]:
# describe（统计概览）/ describe
print(df_pl.describe())


In [ ]:
# 选列：直接传字符串 / Pick columns by name
df_pl.select(["order_id", "unit_price", "quantity"]).head(3)


<a id="4"></a>
## 4. Expression API —— Polars 的核心 / The Core of Polars

**这一节是 Polars 学习曲线最陡的地方，也是它最强大的地方。**
**This is the steepest part of the learning curve, and the most powerful concept.**

Polars 引入了 **Expression**：一个**未执行的运算表达式**，由 `pl.col("xxx")` 起头，可以串接多个变换。
A **Polars Expression** is an **unexecuted computation graph** starting with `pl.col("xxx")`.

为什么这么设计？
Why?
- **可优化**：整个图可以被 query optimizer 重写
- **可并行**：每个独立 expression 可以分到不同线程
- **可重用**：同一个表达式可以用在 `select` / `with_columns` / `filter` / `agg` / `over` 任何地方

### 一个表达式的样子 / What an expression looks like


In [ ]:
# 一个 expression：单价 × 数量 + 5%税 / unit_price × quantity + 5% tax
expr = (pl.col("unit_price") * pl.col("quantity") * 1.05).alias("total_with_tax")
print(expr)        # 注意：没运行！只是个对象 / not executed — just an object
print(type(expr))


In [ ]:
# 把 expression 喂给 DataFrame.select / Plug expressions into select
df_pl.select(
    pl.col("order_id"),
    pl.col("unit_price"),
    pl.col("quantity"),
    expr,                       # 复用上面的 expression
).head(5)


### Expression 的常见操作 / Common operations

| 你想做的 / What you want | 表达式 / Expression |
|---|---|
| 取一列 | `pl.col("x")` |
| 取多列 | `pl.col(["x", "y"])` 或 `pl.col("x", "y")` |
| 取所有数值列 | `pl.col(pl.NUMERIC_DTYPES)` |
| 常量 | `pl.lit(42)` |
| 算术 | `pl.col("a") + pl.col("b")` |
| 比较 | `pl.col("a") > 100` |
| 条件 | `pl.when(cond).then(a).otherwise(b)` |
| 重命名 | `expr.alias("new_name")` |
| 聚合 | `pl.col("a").sum()`, `.mean()`, `.min()`, `.max()` |
| 累计 | `.cum_sum()`, `.cum_max()` |
| 排名 | `.rank()` |
| 类型转换 | `expr.cast(pl.Int64)` |


<a id="5"></a>
## 5. `select` / `with_columns` / `filter` —— 三个最常用方法

| 方法 / Method | 作用 / Purpose |
|---|---|
| `select(exprs...)` | **替换**所有列为新表达式的输出 / **replace** all columns |
| `with_columns(exprs...)` | **添加/覆盖**指定列，其他保留 / **add or overwrite**, keep others |
| `filter(cond)` | 按布尔表达式过滤行 / filter rows |


In [ ]:
# 1) with_columns: 加列，保留原列 / Add columns, keep originals
df_with = df_pl.with_columns(
    total_revenue=pl.col("unit_price") * pl.col("quantity"),
    is_paypal=pl.col("payment_method") == "paypal",
)
df_with.select(["order_id", "unit_price", "quantity",
                "total_revenue", "is_paypal"]).head(3)


In [ ]:
# 2) filter: 多条件 / Multi-condition filter
expensive_us = df_pl.filter(
    (pl.col("country") == "US") &
    (pl.col("unit_price") > 100)
)
print(f"shape: {expensive_us.shape}")
expensive_us.head(3)


In [ ]:
# 3) when / then / otherwise: 条件赋值 / Conditional column
df_priced = df_pl.with_columns(
    price_band=pl.when(pl.col("unit_price") < 20).then(pl.lit("low"))
                 .when(pl.col("unit_price") < 80).then(pl.lit("mid"))
                 .otherwise(pl.lit("high"))
)
df_priced.select(["unit_price", "price_band"]).head(6)


<a id="6"></a>
## 6. 聚合与分组 / Aggregations & `group_by`

`group_by` + `agg` 用 expression 来表达聚合，**比 pandas 灵活得多**。
`group_by` + `agg` uses expressions for aggregation — **far more flexible than pandas**.


In [ ]:
# 简单聚合 / Simple aggregation
overall = df_pl.select(
    n_orders=pl.len(),
    total_revenue=(pl.col("unit_price") * pl.col("quantity")).sum(),
    avg_price=pl.col("unit_price").mean(),
    unique_users=pl.col("user_id").n_unique(),
)
print(overall)


In [ ]:
# 按 category 分组 / Group by category
by_cat = (
    df_pl
    .group_by("category")
    .agg(
        n_orders=pl.len(),
        revenue=(pl.col("unit_price") * pl.col("quantity")).sum(),
        avg_qty=pl.col("quantity").mean(),
        median_price=pl.col("unit_price").median(),
        unique_users=pl.col("user_id").n_unique(),
    )
    .sort("revenue", descending=True)
)
print(by_cat)


In [ ]:
# 多列 group_by + 多种聚合 / Multi-key group_by + multi-agg
by_country_payment = (
    df_pl
    .group_by(["country", "payment_method"])
    .agg(
        n=pl.len(),
        revenue=(pl.col("unit_price") * pl.col("quantity")).sum().round(2),
    )
    .sort(["country", "revenue"], descending=[False, True])
)
print(by_country_payment.head(10))


<a id="7"></a>
## 7. 窗口函数 / Window Functions with `over`

SQL 里的 `OVER (PARTITION BY ...)`. **不改变行数**，每行带上"组内"的某个聚合值。
Equivalent of SQL `OVER (PARTITION BY ...)`. **Preserves row count**, attaches a per-group aggregate.

Polars 用 `.over(...)` 一行写出来——这正是它最爽的地方之一。
Polars expresses this with `.over(...)` — one of its sweet spots.


In [ ]:
# 每个用户的累计消费 / Per-user cumulative spend
out = (
    df_pl
    .sort(["user_id", "order_ts"])
    .with_columns(
        revenue=pl.col("unit_price") * pl.col("quantity"),
    )
    .with_columns(
        user_cum_revenue=pl.col("revenue").cum_sum().over("user_id"),
        user_total_orders=pl.len().over("user_id"),
        user_avg_price=pl.col("unit_price").mean().over("user_id"),
    )
    .select(["user_id", "order_ts", "revenue",
             "user_cum_revenue", "user_total_orders", "user_avg_price"])
    .head(8)
)
print(out)


**关键观察 / Key insight**：上面是**对单个用户连续 8 行的"画像扩展"**——可以看到：
- `user_cum_revenue` 单调上升
- `user_total_orders` 在同一用户内是常数（"该用户一共下了 N 单"）
- `user_avg_price` 也是常数（该用户的平均单价）

用 pandas 这同样能做，但要么用 `groupby().transform()`、要么写 `cumsum`+`merge`，**啰嗦**。Polars 一行 `over()` 搞定。
Pandas can do this too, but needs `groupby().transform()` or `cumsum` + `merge` — Polars does it in one `over()`.


In [ ]:
# 复杂例：每个类目里订单按金额排名 / Rank orders within each category by revenue
ranked = (
    df_pl
    .with_columns(revenue=pl.col("unit_price") * pl.col("quantity"))
    .with_columns(
        cat_rank=pl.col("revenue").rank(method="dense", descending=True).over("category"),
    )
    .filter(pl.col("cat_rank") <= 3)        # 每类前 3 / top-3 per category
    .sort(["category", "cat_rank"])
    .select(["category", "cat_rank", "revenue", "user_id"])
)
print(ranked.head(12))


<a id="8"></a>
## 8. 连接 / Joins


In [ ]:
# 造一个用户维度表 / A user dimension table
users = pl.DataFrame({
    "user_id": np.arange(1, 11, dtype=np.int32),
    "name":    [f"user_{i}" for i in range(1, 11)],
    "tier":    ["bronze","silver","gold","silver","gold",
                "bronze","silver","gold","platinum","platinum"],
})
print(users)


In [ ]:
# Inner join: 只保留两边都有的 / keep matching
joined = (
    df_pl
    .filter(pl.col("user_id") < 11)
    .join(users, on="user_id", how="inner")
    .select(["user_id", "name", "tier", "category", "unit_price", "quantity"])
    .head(5)
)
print(joined)


In [ ]:
# Left join + 处理缺失 / Left join + handle missing
sub = df_pl.head(5)
joined = sub.join(users, on="user_id", how="left")
print(joined.select(["user_id", "name", "tier", "category"]))


<a id="9"></a>
## 9. 长宽转换 / `pivot` and `unpivot`


In [ ]:
# pivot：长 → 宽 / long to wide
# 每个国家 × 支付方式 → 订单数
matrix = (
    df_pl
    .group_by(["country", "payment_method"])
    .agg(n=pl.len())
    .pivot(index="country", on="payment_method", values="n")
    .sort("country")
)
print(matrix)


In [ ]:
# unpivot (pandas 的 melt)：宽 → 长 / wide to long
back_to_long = matrix.unpivot(
    index="country",
    on=["credit", "debit", "paypal", "applepay"],
    variable_name="payment_method",
    value_name="n_orders",
)
print(back_to_long.head(8))


<a id="10"></a>
## 10. 字符串与日期 / Strings and Datetimes

和 pandas 类似的命名空间：`pl.col("x").str.<method>` 和 `pl.col("x").dt.<method>`。
Pandas-like namespaces.


In [ ]:
# 字符串操作 / String ops
demo = pl.DataFrame({
    "email": ["alice@gmail.com", "BOB@YAHOO.COM", "charlie@anthropic.com"],
})

demo.with_columns(
    domain=pl.col("email").str.to_lowercase().str.split("@").list.get(1),
    is_corp=pl.col("email").str.to_lowercase().str.contains("anthropic"),
    char_count=pl.col("email").str.len_chars(),
)


In [ ]:
# 日期/时间操作 / Datetime ops
date_features = (
    df_pl
    .select(
        pl.col("order_ts"),
        pl.col("order_ts").dt.year().alias("year"),
        pl.col("order_ts").dt.month().alias("month"),
        pl.col("order_ts").dt.weekday().alias("weekday"),   # 1=Mon
        pl.col("order_ts").dt.hour().alias("hour"),
        pl.col("order_ts").dt.date().alias("date"),
    )
    .head(5)
)
print(date_features)


In [ ]:
# 按天聚合 / Daily aggregation
daily = (
    df_pl
    .with_columns(date=pl.col("order_ts").dt.date(),
                  revenue=pl.col("unit_price") * pl.col("quantity"))
    .group_by("date")
    .agg(
        n_orders=pl.len(),
        revenue=pl.col("revenue").sum().round(2),
    )
    .sort("date")
)
print(f"#days: {len(daily)}")
print(daily.head(5))


<a id="11"></a>
## 11. Lazy API + Query Plan 优化

**这才是 Polars 真正的"杀手锏"。**
**This is Polars's real superpower.**

Eager 模式每一步立刻执行；**Lazy** 模式把整个查询构建成一棵树，到 `.collect()` 时一次性**优化 + 执行**：

- **谓词下推 / Predicate pushdown**：把 `filter` 推到最早，少处理数据
- **投影下推 / Projection pushdown**：只读你最终要的列
- **公共子表达式消除 / CSE**
- **并行执行 / Parallel execution**

LazyFrame 用 `pl.scan_csv()` / `df.lazy()` 创建。


In [ ]:
# 同一个查询的两种写法对比 / Same query, eager vs lazy
import time

# 1) Eager: 每一步都立即执行
t0 = time.perf_counter()
result_eager = (
    df_pl
    .with_columns(revenue=pl.col("unit_price") * pl.col("quantity"))
    .filter(pl.col("country") == "US")
    .group_by("category")
    .agg(total=pl.col("revenue").sum())
    .sort("total", descending=True)
)
t_eager = time.perf_counter() - t0

# 2) Lazy: 整个图先构建，最后一次性执行
t0 = time.perf_counter()
result_lazy = (
    df_pl.lazy()
    .with_columns(revenue=pl.col("unit_price") * pl.col("quantity"))
    .filter(pl.col("country") == "US")
    .group_by("category")
    .agg(total=pl.col("revenue").sum())
    .sort("total", descending=True)
    .collect()
)
t_lazy = time.perf_counter() - t0

print(f"eager : {t_eager*1000:6.1f} ms")
print(f"lazy  : {t_lazy*1000:6.1f} ms")
print()
print(result_lazy)


In [ ]:
# 查看 query plan / Inspect the optimized query plan
lf = (
    df_pl.lazy()
    .with_columns(revenue=pl.col("unit_price") * pl.col("quantity"))
    .filter(pl.col("country") == "US")
    .group_by("category")
    .agg(total=pl.col("revenue").sum())
)

# 看优化后的计划 / Show optimized plan
print(lf.explain())


读 `explain()` 输出的关键 / How to read `explain()`：
- 看是否把 `FILTER` 推到了 `WITH_COLUMNS` 之前（**Predicate pushdown** 生效）
- 看 `PROJECT */N` 中 `N` 是不是只有你要的几列（**Projection pushdown** 生效）

实际工业场景里，**几乎所有"读 CSV / Parquet → 过滤 → 聚合"流水线都应该用 lazy**，因为 `scan_csv` / `scan_parquet` 配合下推后**根本不会把不要的列/行读入内存**。
In real pipelines, **always use lazy for "read file → filter → agg"** — `scan_csv` / `scan_parquet` + pushdown won't even load the columns/rows you don't need.


<a id="12"></a>
## 12. SQL on Polars

Polars 自带一个 SQL 引擎。**当你脑子里突然想用 SQL 时**——直接写就行，无须切换到 DuckDB / Spark。
Polars ships a SQL engine. **Write SQL whenever your brain reaches for it** — no need to switch to DuckDB / Spark.


In [ ]:
ctx = pl.SQLContext(orders=df_pl)

q = """
SELECT
    country,
    category,
    COUNT(*) AS n_orders,
    ROUND(SUM(unit_price * quantity), 2) AS revenue
FROM orders
WHERE unit_price > 30
GROUP BY country, category
ORDER BY revenue DESC
LIMIT 10
"""

print(ctx.execute(q, eager=True))


<a id="13"></a>
## 13. 性能对比 pandas / Benchmark vs Pandas

用同样的 1,000,000 行数据，做同一个查询：
Same 1,000,000 rows, same query:

> 复杂度：`with_columns` + `filter` + `group_by` + `agg` + `sort`


In [ ]:
import time

# 把 Polars DataFrame 转成 pandas / Convert to pandas
df_pd = df_pl.to_pandas()
print(f"polars size : {df_pl.estimated_size('mb'):.1f} MB")
print(f"pandas size : {df_pd.memory_usage(deep=True).sum()/1024/1024:.1f} MB")


In [ ]:
# Pandas 版本 / pandas version
def pandas_query(df):
    df = df.assign(revenue=df["unit_price"] * df["quantity"])
    df = df[df["country"] == "US"]
    return (df.groupby("category", observed=True)["revenue"]
              .sum().sort_values(ascending=False))

# Polars Eager 版本 / polars eager
def polars_eager_query(df):
    return (df
        .with_columns(revenue=pl.col("unit_price") * pl.col("quantity"))
        .filter(pl.col("country") == "US")
        .group_by("category")
        .agg(total=pl.col("revenue").sum())
        .sort("total", descending=True))

# Polars Lazy 版本 / polars lazy
def polars_lazy_query(df):
    return (df.lazy()
        .with_columns(revenue=pl.col("unit_price") * pl.col("quantity"))
        .filter(pl.col("country") == "US")
        .group_by("category")
        .agg(total=pl.col("revenue").sum())
        .sort("total", descending=True)
        .collect())

# Warm up + time / 预热 + 计时
def benchmark(fn, arg, n_runs=5):
    fn(arg)        # warm up
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn(arg)
        times.append(time.perf_counter() - t0)
    return min(times)   # 取最小（最少噪声）/ best of N

t_pd  = benchmark(pandas_query,        df_pd)
t_pe  = benchmark(polars_eager_query,  df_pl)
t_pl_ = benchmark(polars_lazy_query,   df_pl)

print(f"{'pandas':<14} : {t_pd *1000:7.1f} ms   (×1)")
print(f"{'polars eager':<14} : {t_pe *1000:7.1f} ms   (×{t_pd/t_pe:.1f})")
print(f"{'polars lazy':<14} : {t_pl_*1000:7.1f} ms   (×{t_pd/t_pl_:.1f})")


> 在本机（单数据科学笔记本，1M 行）Polars 通常比 pandas 快 **3–10 倍**；数据上到 10M+ 时差距会拉到 **20–50 倍**。
> On a typical laptop with 1M rows Polars is 3–10× faster than pandas; the gap grows to 20–50× at 10M+ rows.
>
> **Lazy 在小数据上不一定更快**（优化器本身有开销），但在大数据和 IO-bound 场景下优势巨大。
> Lazy isn't always faster on tiny data — the optimizer has overhead — but pays off massively on big / IO-bound workloads.


<a id="14"></a>
## 14. 实战：订单数据 EDA / Hands-on EDA

用 Polars 的 lazy + expression 写一份完整流水线。
A full pipeline with lazy + expressions.


In [ ]:
eda_pipeline = (
    df_pl.lazy()
    # 1) 派生特征 / Derived features
    .with_columns(
        revenue=pl.col("unit_price") * pl.col("quantity"),
        date=pl.col("order_ts").dt.date(),
        hour=pl.col("order_ts").dt.hour(),
        weekday=pl.col("order_ts").dt.weekday(),       # 1=Mon, 7=Sun
    )
)

# 各类目营收 / Revenue by category
by_cat = (
    eda_pipeline
    .group_by("category")
    .agg(
        n_orders=pl.len(),
        revenue=pl.col("revenue").sum().round(2),
        avg_basket=(pl.col("revenue").sum() / pl.len()).round(2),
    )
    .sort("revenue", descending=True)
    .collect()
)
print("=== Revenue by category ===")
print(by_cat)


In [ ]:
# 各国 × 类目 营收矩阵 / Country × category revenue matrix
country_cat = (
    eda_pipeline
    .group_by(["country", "category"])
    .agg(revenue=pl.col("revenue").sum().round(0))
    .collect()
    .pivot(index="country", on="category", values="revenue")
    .sort("country")
)
print("\n=== Country × category revenue ===")
print(country_cat)


In [ ]:
# 用户分布：客单价 vs 订单频次 / Per-user RFM-like stats
per_user = (
    eda_pipeline
    .group_by("user_id")
    .agg(
        n_orders=pl.len(),
        total_spent=pl.col("revenue").sum(),
        avg_basket=pl.col("revenue").mean(),
        unique_categories=pl.col("category").n_unique(),
    )
    .collect()
)

print("=== Per-user summary ===")
print(per_user.describe())


In [ ]:
# 一周内每小时订单数（看消费习惯）/ Orders by weekday × hour
heatmap = (
    eda_pipeline
    .group_by(["weekday", "hour"])
    .agg(n=pl.len())
    .collect()
    .pivot(index="weekday", on="hour", values="n")
    .sort("weekday")
)
print("=== Orders by weekday × hour ===")
print(heatmap)


In [ ]:
# 大客户 Top 10 / Top 10 high-value users
top_users = (
    eda_pipeline
    .group_by("user_id")
    .agg(
        n_orders=pl.len(),
        total_spent=pl.col("revenue").sum().round(2),
    )
    .sort("total_spent", descending=True)
    .head(10)
    .collect()
)
print("=== Top 10 customers ===")
print(top_users)


**总流程**只走了**一次 lazy graph，多次 `.collect()`**——Polars 会复用中间结果，比写一堆 pandas 中间变量高效得多。
**The pipeline runs through one lazy graph with multiple `.collect()` points** — Polars reuses intermediate work, far more efficient than scattering pandas intermediates.


<a id="15"></a>
## 15. 小结 / Summary

| 主题 / Topic | 关键 / Key takeaway |
|---|---|
| Polars 三大快 | 列存 (Arrow) + 多线程 (Rust) + lazy + 优化器 |
| Expression API | `pl.col(x)` 是个**未执行的图节点**；所有方法都接受它 |
| `select` vs `with_columns` | select 替换所有列；with_columns 保留+追加 |
| `filter` | 用 `&`, `|`, `~` 组合布尔 expression |
| `group_by` + `agg` | 用 expression 表达任意聚合，比 pandas 灵活 |
| `over(...)` | 窗口函数：组内累计/排名/扩展，一行搞定 |
| `join` | 接口和 pandas merge 类似 |
| Lazy | 整图优化，谓词/投影下推；**大数据首选** |
| SQL | `pl.SQLContext` 直接写 SQL |
| 性能 | 1M 行 → 3–10×；10M+ 行 → 20–50× |

### 工业场景速查 / Cheat sheet

| 任务 / Task | Polars 写法 |
|---|---|
| 读大 Parquet 只用部分列 | `pl.scan_parquet(path).select([...]).filter(...).collect()` |
| 按组中位数填缺失 | `pl.col("x").fill_null(pl.col("x").median().over("g"))` |
| 计算行号 | `pl.int_range(0, pl.len())` |
| Top-K per group | `.rank().over("g") <= k` |
| Z-score per group | `(pl.col("x") - pl.col("x").mean().over("g")) / pl.col("x").std().over("g")` |
| Rolling 平均 | `pl.col("x").rolling_mean(window_size=7)` |
| 字符串切分取第 i 块 | `pl.col("x").str.split("/").list.get(i)` |
| JSON 拍平 | `pl.col("json_str").str.json_decode().struct.unnest()` |

### 何时选 Polars / When to choose Polars

| 数据规模 / Size | 推荐 / Pick |
|---|---|
| < 100K 行 | 都行 / either; pandas 生态更熟 |
| 100K – 100M 行 | **Polars** |
| 100M – 几个 B 行 | Polars + 分批 / 或 DuckDB |
| > 10B 行 | **Spark / Dask**（Part 21）|

### 下一节预告 / Next up

**Part 0.5 · Matplotlib & Seaborn** —— 数据科学的"标准画板"。我们会用 Iris / Tips 数据集系统讲 figure-axes 模型、分布图、关系图、相关性热力图、定制风格。
**Part 0.5 · Matplotlib & Seaborn** — the standard plotting toolkit. We'll cover the figure-axes model, distribution plots, relationship plots, correlation heatmaps, and styling using Iris / Tips.
